In [ ]:
import os
import time
import pandas as pd
import requests

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------
TICKER = "AAPL"
TARGET_MESSAGES = 600  # Réduit légèrement pour éviter la saturation
OUTPUT_FILE = f"stocktwits_{TICKER.lower()}.csv"

# En-têtes complets pour imiter un vrai navigateur web (évite la 403)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": f"https://stocktwits.com/symbol/{TICKER}",
    "Origin": "https://stocktwits.com",
}

# -------------------------------------------------------------------
# Collecte avec gestion d'erreurs
# -------------------------------------------------------------------
messages = []
max_id = None

print(f"📥 Début de la collecte sécurisée pour ${TICKER}...")

while len(messages) < TARGET_MESSAGES:
  url = f"https://api.stocktwits.com/api/2/streams/symbol/{TICKER}.json"
  params = {}
  if max_id:
    params["max"] = max_id

  try:
    response = requests.get(
        url, params=params, headers=HEADERS, timeout=15
    )

    if response.status_code == 403:
      print(
          "⚠️ Code 403 détecté. Pause de sécurité de 5 secondes avant de"
          " réessayer..."
      )
      time.sleep(5)
      continue
    elif response.status_code == 429:
      print("⚠️ Trop de requêtes. Pause de 30 secondes...")
      time.sleep(30)
      continue
    elif response.status_code != 200:
      print(f"❌ Erreur HTTP {response.status_code}. Arrêt de la collecte.")
      break

    data = response.json()
    batch_messages = data.get("messages", [])

    if not batch_messages:
      print("ℹ️ Plus aucun message disponible.")
      break

    for msg in batch_messages:
      messages.append({
          "id": msg.get("id"),
          "created_at": msg.get("created_at"),
          "body": msg.get("body", ""),
      })

    max_id = batch_messages[-1]["id"]
    print(f"✔️ {len(messages)} / {TARGET_MESSAGES} messages récupérés...")

    # Pause indispensable pour ne pas être banni
    time.sleep(2)

  except Exception as e:
    print(f"❌ Erreur réseau : {e}")
    break

# -------------------------------------------------------------------
# Sauvegarde
# -------------------------------------------------------------------
if messages:
  df = pd.DataFrame(messages)
  df["created_at"] = pd.to_datetime(df["created_at"])
  df = df.sort_values("created_at", ascending=True).reset_index(drop=True)

  os.makedirs("data", exist_ok=True)
  file_path = os.path.join("data", OUTPUT_FILE)
  df.to_csv(file_path, index=False, encoding="utf-8")

  nb_jours = (df["created_at"].max() - df["created_at"].min()).days
  print("\n" + "=" * 50)
  print(f"✅ {len(df)} messages sauvegardés dans '{file_path}'")
  print(f"📅 Couverture : environ {nb_jours} jour(s) d'historique.")
  print("=" * 50)
else:
  print("\n❌ Aucun message récupéré.")

📥 Début de la collecte sécurisée pour $AAPL...
✔️ 30 / 600 messages récupérés...
✔️ 60 / 600 messages récupérés...
✔️ 90 / 600 messages récupérés...
✔️ 120 / 600 messages récupérés...
✔️ 150 / 600 messages récupérés...
✔️ 180 / 600 messages récupérés...
✔️ 210 / 600 messages récupérés...
✔️ 240 / 600 messages récupérés...
✔️ 270 / 600 messages récupérés...
✔️ 300 / 600 messages récupérés...
✔️ 330 / 600 messages récupérés...
✔️ 360 / 600 messages récupérés...
✔️ 390 / 600 messages récupérés...
✔️ 420 / 600 messages récupérés...
✔️ 450 / 600 messages récupérés...
✔️ 480 / 600 messages récupérés...
✔️ 510 / 600 messages récupérés...
✔️ 540 / 600 messages récupérés...
✔️ 570 / 600 messages récupérés...
✔️ 600 / 600 messages récupérés...

✅ 600 messages sauvegardés dans 'data\stocktwits_aapl.csv'
📅 Couverture : environ 4 jour(s) d'historique.
